# Climate TRACE API — Guide

This notebook shows common ways to query the [Climate TRACE API](https://api.climatetrace.org/v7/docs/index.html) using the `climate_trace_tools.api` client shipped with this repository.

The client is a thin wrapper over the API: each function calls one endpoint and returns the parsed JSON (a `dict` or `list`). See [`climate_trace_tools/api/README.md`](../README.md) for the full endpoint reference.

No API key is required.

## Setup

Install the package (from the repo root) if you haven't already:

```bash
pip install git+https://github.com/climatetracecoalition/climate-trace-tools.git
```

In [ ]:
from climate_trace_tools.api import (
    get_aggregate_emissions,
    get_sources,
    get_source,
    rank_countries,
    search_admins,
    search_cities,
    search_owners,
    list_gases,
    list_sectors,
    list_countries,
)

## 1. Discover valid filter values

The `list_*` (definitions) endpoints tell you which values are valid for filters like `gas`, `sectors`, and `continent`. Start here when you're unsure what to pass.

In [ ]:
# Supported gases (use these for the `gas` parameter)
list_gases()

In [ ]:
# Supported sectors (use these for the `sectors` parameter)
list_sectors()

## 2. Aggregate emissions

`get_aggregate_emissions` returns emissions totals for a filtered slice of the data. With no location filter, you get the global total.

In [ ]:
# Global methane (CH4) emissions for 2023
global_ch4_2023 = get_aggregate_emissions(gas='ch4', year=2023)
global_ch4_2023['totals']

### Filtering by sector and region

Filters can be combined. `sectors`/`subsectors` accept either a list or a comma-separated string. Use a continent name from `list_continents()` for the `continent` filter.

In [ ]:
# North America agriculture CH4 emissions for 2023
na_ag_ch4_2023 = get_aggregate_emissions(
    gas='ch4',
    year=2023,
    sectors=['agriculture'],
    continent='North America',
)
na_ag_ch4_2023['totals']

## 3. Look up a country (administrative area)

Most location filters use a `gadm_id` (administrative area id). Use `search_admins` to find one by name. `level=0` restricts the search to countries.

Search endpoints always return a **list**, even when there is a single match.

In [ ]:
# Find Denmark and grab its id
denmark = search_admins(name='Denmark', level=0)[0]
denmark

Alternatively, browse every country with `list_countries()` and pick the id you need.

In [ ]:
countries = list_countries()
countries[:10]

## 4. Country emissions

Pass the country's id as `gadm_id` to scope aggregate emissions to that country.

In [ ]:
# All of Denmark's CO2 emissions in 2023
denmark_co2_2023 = get_aggregate_emissions(gadm_id=denmark['id'], year=2023, gas='co2')
denmark_co2_2023['totals']

In [ ]:
# Narrow to a single sector and pull out the headline number
sector = 'manufacturing'
year = 2023
gas = 'co2'

res = get_aggregate_emissions(gadm_id=denmark['id'], year=year, gas=gas, sectors=sector)
try:
    qty = res['totals']['summaries'][0]['emissionsQuantity']
    print(f"Denmark's {sector} sector emitted {qty} tonnes of {gas} in {year}")
except (KeyError, IndexError) as e:
    print(f'Could not read emissions quantity: {e}')

## 5. Individual sources (assets)

`get_sources` returns the individual emitting sources behind those totals, ranked by emissions. It takes the same filters as `get_aggregate_emissions`, plus `limit`/`offset` for pagination.

In [ ]:
# Top manufacturing CO2 sources in Denmark, 2023
sources = get_sources(
    gadm_id=denmark['id'],
    year=2023,
    gas='co2',
    sectors='manufacturing',
    limit=5,
)
sources

### Drill into one source over time

Given a source id, `get_source` returns details and an emissions time series. Use `start`/`end` and `time_granularity` to control the range and resolution.

In [ ]:
# Emissions time series for the top source above
if sources:
    source_id = sources[0]['id'] if isinstance(sources[0], dict) else sources[0]
    detail = get_source(source_id, start='2021', end='2023', time_granularity='year', gas='co2')
    detail

## 6. Rank countries

`rank_countries` ranks countries by emissions over a time range. `start`/`end` accept a year (`'2023'`), month (`'2023-01'`), or day (`'2023-01-31'`).

In [ ]:
# Countries ranked by power-sector CO2 in 2023
ranking = rank_countries(gas='co2', start='2023', end='2023', sectors='power')
ranking

## 7. Cities and owners

Cities and asset owners have their own search endpoints. Owner ids can then be fed back into emissions queries via `owner_ids`.

In [ ]:
# Search cities by name (optionally scoped to an ISO3 country code)
search_cities(name='Copenhagen', country='DNK', limit=3)

In [ ]:
# Search asset owners by name
search_owners(name='Maersk', limit=3)

## Where to go next

- Full endpoint reference: [`climate_trace_tools/api/README.md`](../README.md)
- Interactive API docs: https://api.climatetrace.org/v7/docs/index.html

Every client function also accepts extra keyword arguments, which are passed straight through as query parameters — so you can use new API parameters even before this client is updated.